In [2]:
import h5py
import hdf5plugin  # <-- This automatically loads the missing decompression plugins!
import matplotlib.pyplot as plt

with h5py.File('data/tworoom.h5', 'r') as f:
    pixel_data = f['pixels']
    action_data = f['action']
    print("Dataset shape:", pixel_data.shape)

    # This step should work smoothly now without the OSError
    first_image = pixel_data[:100]
    first_action = action_data[:100]
def visualize_images_and_actions(images, actions):
    for i in range(images.shape[0]):
        plt.figure(figsize=(8, 8))
        plt.subplot(1, 4, i + 1)
        plt.imshow(images[i])
        plt.axis('off')
        plt.title(f'Action: {actions[i]}')
    plt.show()


Dataset shape: (920809, 224, 224, 3)


In [3]:
import torch
import pandas as pd
with h5py.File("data/tworoom.h5", "r") as f:
    # Load the dataset into a pandas DataFrame
    df = pd.DataFrame({
        'pixels': list(f['pixels'][:512]),
        'actions': list(f['action'][:512])
    })

raw_image = df.iloc[1]['pixels']
raw_image.shape

(224, 224, 3)

In [4]:
o_batch = []
a_batch = []
for i in range(8):
    # make a batch of 8 images
    raw_image = df.iloc[i]['pixels']
    o_batch.append(torch.tensor(raw_image, dtype=torch.float32).T)
    a_batch.append(df.iloc[i]['actions'])

o_batch = torch.stack(o_batch)  # Convert list of tensors to a single tensor
a_batch = torch.tensor(a_batch, dtype=torch.float32)
o_batch.shape, a_batch.shape

/var/folders/ms/b1sqqwtj7zjb6_v_3jgppjnr0000gn/T/ipykernel_19204/1007604671.py:6: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4424.)
  o_batch.append(torch.tensor(raw_image, dtype=torch.float32).T)
/var/folders/ms/b1sqqwtj7zjb6_v_3jgppjnr0000gn/T/ipykernel_19204/1007604671.py:10: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  a_batch = torch.tensor(a_batch, dtype=torch.float32)


(torch.Size([8, 3, 224, 224]), torch.Size([8, 2]))

In [5]:
from encoder import Encoder
from predictor import Predictor
import torch
from torch.nn import MSELoss
from sigreg import SIGReg

# batch of images of tworoom
encoder = Encoder(input_size=192, output_size=192)
pred = Predictor()
loss_fn = MSELoss()
sigreg = SIGReg()


/Users/beta/code/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/beta/code/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

steps = 1000
lr = 0.001
for i in range(steps):
    # calculate loss
    emb = encoder(o_batch)

    z_t_hat = pred(emb, a_batch)
    loss = loss_fn(emb, z_t_hat)
    sigreg_loss = torch.mean(sigreg(emb.transpose(0, 1)))
    loss = loss + 0.1*sigreg_loss

    pred.zero_grad()
    encoder.zero_grad()

    # backward
    loss.backward()

    # zero g
 
    # update weights
    with torch.no_grad():
        for param in pred.parameters():
            param -= lr * param.grad
        for param in encoder.parameters():
            param -= lr * param.grad

    if i % 100 == 0:
        print(f"Step {i}, Loss: {loss.item()}")


Step 0, Loss: 3.3259596824645996
Step 100, Loss: 1.08528470993042


In [ ]:
random_actions = torch.randn(8, 2)
H = 20 # horizon length
o_0 = first_image[:1]
# add a batch dim

emb_0 = encoder(o_batch)[0].unsqueeze(0)
emb_g = encoder(o_batch)[-1].unsqueeze(0)



In [123]:
o_g.shape, o_batch[0].shape

(torch.Size([1, 3, 224, 224]), torch.Size([3, 224, 224]))